# Laboratorio 5: Optimización y selección de modelos en Scikit-learn

**Duración:** 60–90 minutos  
**Objetivo:** practicar estrategias de optimización de modelos:
- **Optimización manual** (train/valid) sobre hiperparámetros.
- **Validación cruzada (CV)** como estimador más estable del rendimiento.
- **GridSearchCV** y **RandomizedSearchCV** para búsqueda sistemática/estocástica.

Usaremos: **Wine (clasificación)** y **Diabetes (regresión)**. Cuando haya preprocesamiento (p.ej., escalado), usaremos **Pipeline** para evitar fuga de información.

## Reglas
- Completa **solo** donde dice `#your code here`
- No cambies nombres de variables/funciones ni orden de retornos.
- Usa `numpy`, `pandas`, `matplotlib`, `scikit-learn`.

In [ ]:
# === Setup (no modificar) ===
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
np.set_printoptions(precision=3, suppress=True)
pd.set_option('display.precision', 3)

from sklearn.datasets import load_wine, load_diabetes
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    mean_squared_error, mean_absolute_error, r2_score
)
from scipy.stats import randint as sp_randint

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print('Setup OK')
import sklearn, sys
print('Python', sys.version.split()[0], '| numpy', np.__version__, '| pandas', pd.__version__, '| sklearn', sklearn.__version__)

In [ ]:
# Datos 
wine = load_wine()
Xw = pd.DataFrame(wine.data, columns=wine.feature_names)
yw = pd.Series(wine.target, name='target')

diab = load_diabetes()
Xd = pd.DataFrame(diab.data, columns=diab.feature_names)
yd = pd.Series(diab.target, name='target')
print('Wine:', Xw.shape, '| Diabetes:', Xd.shape)

## Parte A — Optimización manual (train/valid)
### A.1 Tarea 1 — Split train/valid para Wine
Implementa `split_train_valid_wine(test_size=0.2, valid_size=0.2, scale=True)` que devuelva:
`(X_train, X_valid, X_test, y_train, y_valid, y_test)`.

**Detalles:**
- Primero separa `test` con `test_size` y `random_state=RANDOM_STATE`, estratificando por `yw`.
- Luego separa `valid` desde el *train restante* con `valid_size` (estratificado).
- Si `scale=True`, ajusta `StandardScaler` en `X_train` **y transforma** `X_train`, `X_valid`, `X_test`.

In [ ]:
def split_train_valid_wine(test_size=0.2, valid_size=0.2, scale=True):
    # your code here
    raise NotImplementedError

In [ ]:
# Sanity check visible
Xa_tr, Xa_va, Xa_te, ya_tr, ya_va, ya_te = split_train_valid_wine()
assert Xa_tr.shape[0] + Xa_va.shape[0] + Xa_te.shape[0] == Xw.shape[0]
print('OK split train/valid/test (Wine)')

### A.2 Tarea 2 — Búsqueda manual de `k` en KNN (validación)
Implementa `manual_search_knn_wine(ks, X_train, y_train, X_valid, y_valid)` que:
- Entrene `KNeighborsClassifier(n_neighbors=k)` para cada `k` en `ks`.
- Calcule `accuracy` y `f1_macro` en el *validation set*.
- Devuelva un `DataFrame` con columnas `['k','accuracy','f1_macro']` ordenado por `f1_macro` descendente.

In [ ]:
def manual_search_knn_wine(ks, X_train, y_train, X_valid, y_valid):
    rows = []
    for k in ks:
        # Entrena KNN con k, guarda k, acc y f1 en rows
        acc = 0
        f1 = 0
        # your code here
        raise NotImplementedError
        rows.append([int(k), acc, f1])
    df = pd.DataFrame(rows, columns=['k','accuracy','f1_macro']).sort_values('f1_macro', ascending=False)
    return df

In [ ]:
# Sanity check visible
ks = list(range(1, 21))
df_valid = manual_search_knn_wine(ks, Xa_tr, ya_tr, Xa_va, ya_va)
assert {'k','accuracy','f1_macro'}.issubset(df_valid.columns)
print('OK manual search df')

### A.3 Tarea 3 — Selección del mejor `k` y evaluación en test
Implementa `select_and_eval_best_k(df_valid, X_train, y_train, X_test, y_test)` que:
- Obtenga el **mejor `k`** según `f1_macro`.
- Entrene `KNeighborsClassifier(n_neighbors=best_k)` en **train+valid combinados** (puedes concatenar) y evalúe en `test`.
- Retorne `(best_k, accuracy_test, f1_macro_test)`.

In [ ]:
def select_and_eval_best_k(df_valid, X_train, y_train, X_test, y_test):
    best_k = 0
    acc = 0
    f1 = 0
    # your code here
    raise NotImplementedError
    return best_k, acc, f1


In [ ]:
# Prepara train_full (train+valid) para evaluación final
Xa_full = np.vstack([Xa_tr, Xa_va])
ya_full = np.concatenate([ya_tr, ya_va])
best_k, acc_t, f1_t = select_and_eval_best_k(df_valid, Xa_full, ya_full, Xa_te, ya_te)
print(f'Mejor k={best_k} | Test: acc={acc_t:.3f}, f1_macro={f1_t:.3f}')

## Parte B — GridSearchCV (no evaluada)
### GridSearchCV para KNN en Wine (con Pipeline)
Ahora crearemos `grid_search_knn_wine()` que:
- Use `Pipeline([('scaler', StandardScaler()), ('knn', KNeighborsClassifier())])`.
- Defina un `param_grid` para `knn__n_neighbors` (1..30) y `knn__weights` en `['uniform','distance']`.
- Use `cv=5`, `scoring='f1_macro'`, `n_jobs=-1`.
- Retorne `best_params_, best_score_`.
*Nota:* Pipeline evita fuga de información al escalar **dentro** de cada fold.

In [ ]:
def grid_search_knn_wine():
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('knn', KNeighborsClassifier())
    ])
    param_grid = {
        'knn__n_neighbors': list(range(1,31)),
        'knn__weights': ['uniform','distance']
    }
    gs = GridSearchCV(pipe, param_grid=param_grid, cv=5, scoring='f1_macro', n_jobs=-1)
    gs.fit(Xw.values, yw.values)
    return gs.best_params_, gs.best_score_

In [ ]:
bp, bs = grid_search_knn_wine()
print('Best params:', bp, '| best f1_macro (CV mean):', round(bs,3))

## Parte C — RandomizedSearchCV (regresión, no evaluada)
### RandomizedSearchCV para KNNRegressor en Diabetes (con Pipeline)
Ahora implementaremos `random_search_knr_diabetes()` que:
- Use `Pipeline([('scaler', StandardScaler()), ('knr', KNeighborsRegressor())])`.
- Distribuciones: `knr__n_neighbors ~ randint(1, 31)`, `knr__weights ∈ ['uniform','distance']`, `knr__p ~ randint(1, 3)`.
- `n_iter=20`, `cv=5`, `scoring='neg_mean_squared_error'`, `random_state=RANDOM_STATE`, `n_jobs=-1`.
- Devuelve `best_params_, best_score_` (nota: el *score* será negativo por convención).

In [ ]:
def random_search_knr_diabetes():
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('knr', KNeighborsRegressor())
    ])
    param_distributions = {
        'knr__n_neighbors': sp_randint(1, 31),
        'knr__weights': ['uniform','distance'],
        'knr__p': sp_randint(1, 3)
    }
    rs = RandomizedSearchCV(
        pipe, param_distributions=param_distributions, n_iter=20,
        cv=5, scoring='neg_mean_squared_error', random_state=RANDOM_STATE, n_jobs=-1
    )
    rs.fit(Xd.values, yd.values)
    return rs.best_params_, rs.best_score_

bp_r, bs_r = random_search_knr_diabetes()
print('Best params (Randomized):', bp_r, '| best neg-MSE (CV mean):', round(bs_r,3))

## Parte D — Comparación con validación cruzada (no evaluada)
### Función de comparación Cross Validation
Implementaremos finalmente `cv_compare_models_diabetes()` que:
- Construya dos *pipelines*: `LinearRegression` (con o sin scaler) y `KNeighborsRegressor(k=bp_r['knr__n_neighbors'])` con `StandardScaler`.
- Para cada modelo, calcula `cross_val_score` con `cv=5` para `scoring='neg_mean_squared_error'` **y** `scoring='r2'`.
- Devuelve un `DataFrame` con filas=nombre del modelo y columnas `['neg_MSE_mean','neg_MSE_std','R2_mean','R2_std']`.

In [ ]:
def cv_compare_models_diabetes():
    pipe_lr = Pipeline([('scaler', StandardScaler()), ('lr', LinearRegression())])
    pipe_knr = Pipeline([('scaler', StandardScaler()), ('knr', KNeighborsRegressor(n_neighbors=int(bp_r['knr__n_neighbors'])))])

    mse_lr = cross_val_score(pipe_lr, Xd.values, yd.values, cv=5, scoring='neg_mean_squared_error', n_jobs=-1)
    r2_lr  = cross_val_score(pipe_lr, Xd.values, yd.values, cv=5, scoring='r2', n_jobs=-1)

    mse_kn = cross_val_score(pipe_knr, Xd.values, yd.values, cv=5, scoring='neg_mean_squared_error', n_jobs=-1)
    r2_kn  = cross_val_score(pipe_knr, Xd.values, yd.values, cv=5, scoring='r2', n_jobs=-1)

    df = pd.DataFrame({
        'model': ['LinearRegression', 'KNeighborsRegressor'],
        'neg_MSE_mean': [mse_lr.mean(), mse_kn.mean()],
        'neg_MSE_std':  [mse_lr.std(),  mse_kn.std()],
        'R2_mean':      [r2_lr.mean(),  r2_kn.mean()],
        'R2_std':       [r2_lr.std(),   r2_kn.std()],
    }).set_index('model')
    return df

df_cv = cv_compare_models_diabetes()
assert set(df_cv.columns) == {'neg_MSE_mean','neg_MSE_std','R2_mean','R2_std'}
print('OK CV compare')
print(df_cv)

## Takeaways del laboratorio
- **Manual vs CV**: la validación simple puede sesgar; **CV** estabiliza la estimación.
- **Grid vs Randomized**: Grid explora sistemático; Randomized suele encontrar buenos valores más rápido en espacios amplios.
- **Pipeline**: evita fugas al incluir preprocesamiento dentro de cada fold.

Reinicia y ejecuta **Run All** antes de entregar.